# Dataset Amazon – Camino 1: Forecasting de Ventas

Objetivo: predecir las **unidades encargadas del mes siguiente** para cada producto (ASIN), usando un modelo panel XGBoost con features de retardo y estacionalidad.

Base: `df_total_limpio.csv` generado en el notebook de preprocesamiento.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')


## 2. Carga del dataset limpio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/DatasetAmazon/df_total_limpio.csv',
                 parse_dates=['fecha'])

df = df.sort_values(['ASIN (child)', 'fecha']).reset_index(drop=True)

print(f"Filas: {len(df):,}  |  Productos: {df['ASIN (child)'].nunique()}  |  "
      f"Rango: {df['fecha'].min().date()} → {df['fecha'].max().date()}")


## 3. Feature Engineering


Creamos todas las features dentro de cada ASIN (groupby) para no contaminar un producto con datos de otro:

- **Lags**: unidades del mes anterior (`lag_1`), hace 2 meses (`lag_2`), hace 12 meses (`lag_12`).

- **Medias móviles**: media de los últimos 3 meses (`ma_3`) y 6 meses (`ma_6`).

- **Tasa de conversión**: `Unidades encargadas / Sesiones: total` — captura eficiencia de venta.

- **Sesiones** (tráfico): `Sesiones: total`. Se elimina `Visitas: total` por ser redundante (r > 0.95).

- **Estacionalidad**: mes del año y trimestre como variables cíclicas (sin y cos para que el modelo entienda que enero y diciembre son contiguos).

- **Target**: `Unidades encargadas` del mes siguiente (`target_t1`), creado con `.shift(-1)` dentro de cada ASIN.


⚠️ Las filas donde algún lag o el target sean NaN se descartan — es inevitable con pocos meses de histórico.

In [ ]:
# Aseguramos tipos correctos
df['Unidades encargadas'] = pd.to_numeric(df['Unidades encargadas'], errors='coerce')
df['Sesiones: total']     = pd.to_numeric(df['Sesiones: total'],     errors='coerce')

# ── Features dentro de cada ASIN ──────────────────────────────────────────────
g = df.groupby('ASIN (child)')

# Lags
df['lag_1']  = g['Unidades encargadas'].shift(1)
df['lag_2']  = g['Unidades encargadas'].shift(2)
df['lag_12'] = g['Unidades encargadas'].shift(12)

# Medias móviles (calculadas sobre lag para evitar leakage)
df['ma_3'] = g['Unidades encargadas'].transform(lambda x: x.shift(1).rolling(3).mean())
df['ma_6'] = g['Unidades encargadas'].transform(lambda x: x.shift(1).rolling(6).mean())

# Tasa de conversión (mes actual, que sí conocemos en el momento de predecir)
df['tasa_conversion'] = df['Unidades encargadas'] / df['Sesiones: total'].replace(0, np.nan)

# Mes numérico y estacionalidad cíclica
df['mes_num']  = df['fecha'].dt.month
df['mes_sin']  = np.sin(2 * np.pi * df['mes_num'] / 12)
df['mes_cos']  = np.cos(2 * np.pi * df['mes_num'] / 12)
df['trimestre'] = df['fecha'].dt.quarter

# Target: unidades del MES SIGUIENTE
df['target_t1'] = g['Unidades encargadas'].shift(-1)

# ── Descartamos filas con NaN en features o target ────────────────────────────
feature_cols = ['lag_1', 'lag_2', 'lag_12', 'ma_3', 'ma_6',
                'tasa_conversion', 'Sesiones: total',
                'mes_sin', 'mes_cos', 'trimestre']

df_model = df.dropna(subset=feature_cols + ['target_t1']).copy()

print(f"Filas tras eliminar NaN: {len(df_model):,}  (de {len(df):,} originales)")
print(f"Productos con datos suficientes: {df_model['ASIN (child)'].nunique()}")


## 4. Partición Train / Test — Split Temporal


**Nunca aleatoria en series temporales.** Entrenar con meses recientes y evaluar con meses pasados filtraría información del futuro al pasado (data leakage).


Estrategia: los últimos **3 meses** de cada producto van a test, el resto a train.

In [ ]:
# Fecha de corte: 3 meses antes del último mes del dataset
fecha_corte = df_model['fecha'].max() - pd.DateOffset(months=3)

train = df_model[df_model['fecha'] <= fecha_corte].copy()
test  = df_model[df_model['fecha'] >  fecha_corte].copy()

print(f"Train: {len(train):,} filas  |  {train['fecha'].min().date()} → {train['fecha'].max().date()}")
print(f"Test:  {len(test):,}  filas  |  {test['fecha'].min().date()} → {test['fecha'].max().date()}")

X_train = train[feature_cols]
y_train = train['target_t1']

X_test  = test[feature_cols]
y_test  = test['target_t1']


## 5. Baseline: Predicción Naive


Antes de entrenar cualquier modelo, establecemos un **baseline** — la predicción más simple posible.

Aquí usamos *"el mes que viene venderé lo mismo que el mes pasado"* (`lag_1`). Si nuestro modelo no supera esto, no aporta nada.

In [ ]:
y_pred_baseline = X_test['lag_1']

mae_base  = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_base   = r2_score(y_test, y_pred_baseline)

print("── Baseline (lag_1 = mismo valor mes anterior) ──")
print(f"  MAE  : {mae_base:.2f} unidades")
print(f"  RMSE : {rmse_base:.2f} unidades")
print(f"  R²   : {r2_base:.4f}")


## 6. Modelo: XGBoost Regressor


Usamos **XGBoost** en modo panel: un único modelo para todos los productos, con las features de retardo como contexto.

Ventajas sobre un modelo por producto:
- Aprovecha patrones comunes entre productos (estacionalidad, tendencia).
- Funciona aunque algún producto tenga pocos meses de histórico.
- Más robusto y generalizable.

In [ ]:
model = XGBRegressor(
    n_estimators    = 500,
    learning_rate   = 0.05,
    max_depth       = 5,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    random_state    = 42,
    n_jobs          = -1,
    verbosity       = 0
)

model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          verbose=False)

y_pred = model.predict(X_test)
y_pred = np.maximum(y_pred, 0)  # las unidades no pueden ser negativas

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("── XGBoost ──────────────────────────────────────")
print(f"  MAE  : {mae:.2f} unidades   (baseline: {mae_base:.2f})")
print(f"  RMSE : {rmse:.2f} unidades  (baseline: {rmse_base:.2f})")
print(f"  R²   : {r2:.4f}             (baseline: {r2_base:.4f})")
print()
mejora_mae = (mae_base - mae) / mae_base * 100
print(f"  Mejora sobre baseline (MAE): {mejora_mae:+.1f}%")


## 7. Importancia de Variables

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values()

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='steelblue')
plt.title('Importancia de variables – XGBoost')
plt.xlabel('F-score (gain)')
plt.tight_layout()
plt.show()


## 8. Predicciones vs. Valores Reales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter real vs predicho
axes[0].scatter(y_test, y_pred, alpha=0.4, s=15, color='steelblue')
lim = max(y_test.max(), y_pred.max()) * 1.05
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Unidades reales')
axes[0].set_ylabel('Unidades predichas')
axes[0].set_title('Real vs. Predicho (test)')
axes[0].legend()

# Residuos
residuos = y_test.values - y_pred
axes[1].hist(residuos, bins=40, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Error (real − predicho)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Residuos')

plt.tight_layout()
plt.show()


## 9. Evolución Temporal: Real vs. Predicho (producto ejemplo)

Visualizamos la serie de un producto concreto para ver cómo sigue el modelo la dinámica real.

In [ ]:
# Producto con más observaciones en test
asin_ejemplo = (test.groupby('ASIN (child)')['fecha'].count()
                .idxmax())

idx = test['ASIN (child)'] == asin_ejemplo
fechas_test = test.loc[idx, 'fecha']
real_test   = y_test[idx]
pred_test   = y_pred[test.index.get_indexer(test.loc[idx].index)]

# Histórico de train para contexto
idx_train = train['ASIN (child)'] == asin_ejemplo

plt.figure(figsize=(13, 5))
plt.plot(train.loc[idx_train, 'fecha'], train.loc[idx_train, 'target_t1'],
         marker='o', color='steelblue', label='Train (real)')
plt.plot(fechas_test, real_test.values,
         marker='o', color='green', label='Test (real)')
plt.plot(fechas_test, pred_test,
         marker='s', linestyle='--', color='red', label='Test (predicho)')
plt.axvline(fecha_corte, color='gray', linestyle=':', linewidth=1.5, label='Corte train/test')
plt.title(f'Serie temporal – ASIN: {asin_ejemplo}')
plt.xlabel('Fecha')
plt.ylabel('Unidades encargadas')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 10. Derivación de Ventas en € (variable secundaria)


Una vez tenemos la predicción de unidades, derivamos las ventas en € usando el **ticket medio histórico** de cada producto:

```
ticket_medio = Ventas totales / Unidades totales
Ventas_predichas (€) = Unidades_predichas × ticket_medio
```

In [ ]:
# Ticket medio por ASIN (calculado solo con datos de train para no filtrar test)
ticket_medio = (
    train.groupby('ASIN (child)')
    .apply(lambda x: x['Ventas de productos encargados'].sum() /
                     x['Unidades encargadas'].replace(0, np.nan).sum())
    .rename('ticket_medio')
    .reset_index()
)

# Unir al test
resultados = test[['ASIN (child)', 'fecha', 'target_t1']].copy()
resultados['pred_unidades'] = y_pred
resultados = resultados.merge(ticket_medio, on='ASIN (child)', how='left')
resultados['pred_ventas_eur'] = resultados['pred_unidades'] * resultados['ticket_medio']

# Vista resumen
print("Muestra de predicciones con ventas derivadas (€):")
display(resultados[['ASIN (child)', 'fecha', 'target_t1',
                     'pred_unidades', 'ticket_medio', 'pred_ventas_eur']].head(15).round(2))


## 11. Guardado de Resultados

In [ ]:
resultados.to_csv('/content/drive/MyDrive/DatasetAmazon/predicciones_forecasting.csv', index=False)
print("✅ Predicciones guardadas en predicciones_forecasting.csv")
